In [1]:
from articles_dataset import ArticlesDataset

articles_dataset_path = \
    "../resources/bcc-news-entertainment.csv"
dataset = ArticlesDataset(articles_dataset_path)
dataset.load_articles(50)
dataset.preprocess()

sents_count = 0
for article in dataset.articles:
    sents_count += len(list(article.document.sents))
print("Total sentences count acrros loaded documents: ", sents_count)

Total sentences count acrros loaded documents:  743


In [2]:
import re
import hashlib


def readable_id(sentence: str, length: int = 4) -> str:
    h = hashlib.md5(sentence.encode()).hexdigest()[:length]
    slug = re.sub(r'\W+', '_', sentence.lower()).strip('_')[:30]

    return f"{slug}_{h}"

### NLTK based

In [ ]:
import nltk
nltk.download('maxent_ne_chunker')

In [39]:
from nltk.tree import Tree

def get_continuous_chunks(chunked):
    continuous_chunk = []
    current_chunk = []
    for i in chunked:
        if type(i) == Tree:
            if hasattr(i, 'label') and not i.label() == 'NE':
                current_chunk.append(i.label())
            current_chunk.append(" ".join([token for token, pos in i.leaves()]))
        if current_chunk:
            named_entity = " ".join(current_chunk)
            if named_entity not in continuous_chunk:
                continuous_chunk.append(named_entity)
                current_chunk = []
        else:
            continue

    return continuous_chunk

In [48]:
import os
import pandas as pd
# or load data by: nltk_ner.from_csv("../results/binary_ne_nltk.csv", index=False)
if os.path.exists("../results/binary_ne_nltk.csv") :
    nltk_ner = pd.read_csv("../results/binary_ne_nltk.csv")
else:
    rows = []
    for article in dataset.articles:
        for sentence in article.document.sents:
            # tokens = nltk.word_tokenize(sentence.text)
            tokens = [t.text for t in sentence]
            tagged = nltk.pos_tag(tokens)
            ne_chunked = nltk.ne_chunk(tagged, binary=True)
            continuous_chunk = get_continuous_chunks(ne_chunked)
            rows.append({
                "sentence_id": readable_id(sentence.text),
                "sentence": sentence.text,
                "tokens": tokens,
                "ne": continuous_chunk,
            })

    nltk_ner = pd.DataFrame(rows)

In [49]:
nltk_ner

,sentence_id,sentence,tokens,ne
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,"[' ', 'French', 'musician', 'Jean', '-', 'Mich...","['French', 'Jean', 'Michel Jarre', 'Copenhagen']"
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,"['Denmark', 'is', 'holding', 'a', 'three', '-'...","['Denmark', 'Parken']"
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,"['Other', 'stars', 'are', 'expected', 'to', 'j...",['Danish']
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...","['""', 'Christian', 'Andersen', ""'s"", 'fairy', ...","['Christian Andersen', 'Jarre']"
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...","['""', 'For', 'all', 'of', 'us', ',', 'at', 'an...",[]
...,...,...,...,...
738,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,"['If', 'I', ""'m"", 'half', 'the', 'person', 'yo...",[]
739,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,"['Both', 'Foxx', 'and', 'Swank', 'are', 'now',...","['Swank', 'Oscars', 'Hollywood']"
740,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...","['However', ',', 'Swank', 'has', 'to', 'overco...","['Swank', 'Annette Bening']"
741,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...","['Meanwhile', ',', 'ballots', 'for', 'Oscars',...","['Oscars', 'Hollywood', 'Academy', 'Motion Pic..."


### Hugging face (Jean-Baptiste/roberta-large-ner-english)

In [57]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
# tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER") -- not good enough, has problems with tokenization as well aggregation of named entities
# model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
tokenizer = AutoTokenizer.from_pretrained("Jean-Baptiste/roberta-large-ner-english")
model = AutoModelForTokenClassification.from_pretrained("Jean-Baptiste/roberta-large-ner-english")
nlp = pipeline('ner', model=model, tokenizer=tokenizer, aggregation_strategy="simple")

config.json:   0%|          | 0.00/849 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: Jean-Baptiste/roberta-large-ner-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [61]:
import os
import pandas as pd
# or load data by: nltk_ner.from_csv("../results/binary_ne_nltk.csv", index=False)
rows = []
for article in dataset.articles:
    for sentence in article.document.sents:
        raw_ne = nlp(sentence.text)
        ne = [ne["word"] for ne in raw_ne]
        rows.append({
            "sentence_id": readable_id(sentence.text),
            "sentence": sentence.text,
            "tokens": None,
            "ne": ne,
        })

model_ner = pd.DataFrame(rows)

In [62]:
model_ner

,sentence_id,sentence,tokens,ne
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,None,"[ French, Jean-Michel Jarre, Copenhagen, Ha..."
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,None,"[ Denmark, Parken]"
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,None,[ Danish]
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...",None,"[Christian Andersen, Jarre]"
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...",None,[]
...,...,...,...,...
738,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,None,[]
739,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,None,"[ Foxx, Swank, Oscars, Hollywood]"
740,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...",None,"[ Swank, Annette Bening, Being Julia]"
741,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...",None,"[ Oscars, Hollywood, Academy of Motion Pictu..."


In [63]:
df = model_ner.merge(
    nltk_ner,
    on="sentence_id",
    suffixes=("_model", "_nltk")
)
df = df.drop(columns=["sentence_nltk", "tokens_model"])
df = df.rename(columns={"sentence_model": "sentence"})

In [64]:
df

,sentence_id,sentence,ne_model,tokens_nltk,ne_nltk
0,french_musician_jean_michel_ja_702a,French musician Jean-Michel Jarre is to perfo...,"[ French, Jean-Michel Jarre, Copenhagen, Ha...","[' ', 'French', 'musician', 'Jean', '-', 'Mich...","['French', 'Jean', 'Michel Jarre', 'Copenhagen']"
1,denmark_is_holding_a_three_day_22dc,Denmark is holding a three-day celebration of ...,"[ Denmark, Parken]","['Denmark', 'is', 'holding', 'a', 'three', '-'...","['Denmark', 'Parken']"
2,other_stars_are_expected_to_jo_f264,Other stars are expected to join the line-up i...,[ Danish],"['Other', 'stars', 'are', 'expected', 'to', 'j...",['Danish']
3,christian_andersen_s_fairy_tal_0356,"""Christian Andersen's fairy tales are timeless...","[Christian Andersen, Jarre]","['""', 'Christian', 'Andersen', ""'s"", 'fairy', ...","['Christian Andersen', 'Jarre']"
4,for_all_of_us_at_any_age_there_1692,"""For all of us, at any age there is always - b...",[],"['""', 'For', 'all', 'of', 'us', ',', 'at', 'an...",[]
...,...,...,...,...,...
744,if_i_m_half_the_person_you_are_1bf8,If I'm half the person you are and half the ta...,[],"['If', 'I', ""'m"", 'half', 'the', 'person', 'yo...",[]
745,both_foxx_and_swank_are_now_co_34ae,Both Foxx and Swank are now considered to be a...,"[ Foxx, Swank, Oscars, Hollywood]","['Both', 'Foxx', 'and', 'Swank', 'are', 'now',...","['Swank', 'Oscars', 'Hollywood']"
746,however_swank_has_to_overcome__6e3c,"However, Swank has to overcome a strong challe...","[ Swank, Annette Bening, Being Julia]","['However', ',', 'Swank', 'has', 'to', 'overco...","['Swank', 'Annette Bening']"
747,meanwhile_ballots_for_oscars_t_728e,"Meanwhile, ballots for Oscars - the Hollywood'...","[ Oscars, Hollywood, Academy of Motion Pictu...","['Meanwhile', ',', 'ballots', 'for', 'Oscars',...","['Oscars', 'Hollywood', 'Academy', 'Motion Pic..."


In [65]:
df.to_csv("../results/binary_ne_nltk_vs_model.csv", index=False)